In [1]:
import pandas as pd
from datetime import datetime
import os

In [11]:
# Load the CSV file into a pandas DataFrame
file_path = os.path.realpath('./Data/Combined_Terminal.csv')
print('File: ', file_path)
#allData= pd.read_csv(file_path)
allData= pd.read_csv(file_path, parse_dates=['DateTime'], date_format='%Y-%m-%d %H:%M:%S')
allData.set_index('DateTime', drop = False, inplace=True)
#print(allData.head)
#print(allData.index.name)
allData['Delta'] = 0
# Adding a 'label' column to differentiate between OffPeak, OnPeak, and OnPeakHigh 
allData['label'] = 'OffPeak'
print(allData.head())
#print(allData['DateTime'].iloc[0], allData['DateTime'].iloc[1])
#print(allData['KWH11'].iloc[0], allData['KWH11'].iloc[1])

File:  /home/homayoun/Dropbox/Projects/VisualCode/Python/CAE/New/KWH_based/Data/Combined_Terminal.csv
                               DateTime  KWH11    KW  Year  Delta    label
DateTime                                                                  
2014-07-01 00:14:00 2014-07-01 00:14:00  357.5  1430  2014      0  OffPeak
2014-07-01 00:29:00 2014-07-01 00:29:00  362.0  1448  2014      0  OffPeak
2014-07-01 00:44:00 2014-07-01 00:44:00  350.5  1402  2014      0  OffPeak
2014-07-01 00:59:00 2014-07-01 00:59:00  344.5  1378  2014      0  OffPeak
2014-07-01 01:14:00 2014-07-01 01:14:00  324.5  1298  2014      0  OffPeak


In [14]:
# Find irregular data
def missing_data(data):
    e_td = pd.to_timedelta('15 minutes')
    for i in range(0, len(data)-1, 1):
        c_td = data['DateTime'].iloc[i+1] - data['DateTime'].iloc[i]
        if(e_td != c_td): 
           print (data['DateTime'].iloc[i], data['DateTime'].iloc[i+1], c_td)

missing_data(allData)

2014-11-02 01:44:00 2014-11-02 00:59:00 -1 days +23:15:00
2015-03-08 01:44:00 2015-03-08 02:59:00 0 days 01:15:00
2015-11-01 01:44:00 2015-11-01 00:59:00 -1 days +23:15:00
2016-03-13 01:44:00 2016-03-13 02:59:00 0 days 01:15:00
2016-11-06 01:44:00 2016-11-06 00:59:00 -1 days +23:15:00
2017-03-12 01:44:00 2017-03-12 02:59:00 0 days 01:15:00
2017-11-05 01:44:00 2017-11-05 00:59:00 -1 days +23:15:00
2018-03-11 01:44:00 2018-03-11 02:59:00 0 days 01:15:00
2018-11-04 01:44:00 2018-11-04 00:59:00 -1 days +23:15:00
2019-03-10 01:44:00 2019-03-10 02:59:00 0 days 01:15:00
2019-11-03 01:44:00 2019-11-03 00:59:00 -1 days +23:15:00
2020-03-08 01:44:00 2020-03-08 02:59:00 0 days 01:15:00
2020-11-01 01:44:00 2020-11-01 00:59:00 -1 days +23:15:00
2021-03-14 01:44:00 2021-03-14 02:59:00 0 days 01:15:00
2021-11-07 01:44:00 2021-11-07 00:59:00 -1 days +23:15:00
2022-03-13 01:44:00 2022-03-13 02:59:00 0 days 01:15:00
2022-11-06 01:44:00 2022-11-06 00:59:00 -1 days +23:15:00
2023-03-12 01:44:00 2023-03-12

In [16]:
# Calculate the derivative of the usage
def calc_delta(data):
    e_td = pd.to_timedelta('15 minutes')
    delta = []
    for i in range(0, len(data)-1, 1):
        dUsage = data['KWH11'].iloc[i+1] - data['KWH11'].iloc[i]
        dt = data['DateTime'].iloc[i+1] - data['DateTime'].iloc[i]
        delta.append(dUsage / dt.total_seconds())
    delta.append(0)
    return delta 

allData['Delta'] = calc_delta(allData)

In [17]:
# allData.describe()
allData['KWH11'].describe()

count    368264.000000
mean        209.034644
std          56.015471
min           0.000000
25%         167.000000
50%         200.000000
75%         244.500000
max         447.500000
Name: KWH11, dtype: float64

In [18]:
# This needs closer inspection, but it seems to be a way to set the labels based on the month and hour

def set_labels(data):
    data['label'] = 'w'  # Default label as winter
    month_mask = data.index.month.isin([6, 7, 8, 9])  # June to September
    data.loc[month_mask, 'label'] = 's' # Summer months

    # Identify OnPeak hours in summer
    time_mask = data.index.hour.isin([13, 14, 15, 16, 17, 18, 19, 20, 21])  # Summer OnPeak hours. Does not include 22
    label_mask = data['label'] == 's'
    data.loc[time_mask & label_mask, 'label'] = 'sOnPeak'  # Summer OnPeak

    # Identify OnPeak hours in summer transition months (May and October) 
    time_mask = data.index.hour.isin([13, 14, 15, 16, 17, 18, 19, 20, 21])  # OnPeak hours. Does not include 22
    month_mask = data.index.month.isin([5, 10])  # May or October
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask & month_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify OnPeak hours on deep winter months (November through April) 
    time_mask = data.index.hour.isin([6, 7, 8, 9, 10, 11, 17, 18, 19, 20, 21])  # OnPeak hours. Does not include 22
    # month_mask = data.index.month.isin([5, 10])  # Not needed for the last modification. All remaining are w 
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify the OffPeak times
    label_mask = data['label'] == 's'
    data.loc[label_mask, 'label'] = 'sOffPeak'  # Summer OffPeak 
    label_mask = data['label'] == 'w'
    data.loc[label_mask, 'label'] = 'wOffPeak'  # Winter OffPeak
    label_mask = data.index.day_of_week.isin([5, 6])  # Saturday and Sunday
    data.loc[label_mask, 'label'] = 'wkndOffPeak'  # Weekend OffPeak

    #Identify the high usage periods during OnPeak times
    data.loc[(data['label'] == 'wOnPeak') & (data['KWH11'] > 200), 'label'] = 'wOnPeakHigh'
    data.loc[(data['label'] == 'sOnPeak') & (data['KWH11'] > 200), 'label'] = 'sOnPeakHigh'
    return data

allData = set_labels(allData)
print('allData: ', allData.value_counts('label'), '\n')

allData:  label
wkndOffPeak    105220
wOffPeak        86284
wOnPeak         64576
sOffPeak        56280
wOnPeakHigh     22136
sOnPeak         19730
sOnPeakHigh     14038
Name: count, dtype: int64 



In [19]:
allData.to_csv('Combined_Terminal_Label.csv', index=False)